# Leuven collective borefield — back-of-envelope sizing

Sizes one large, collective borehole field for Leuven centrum's heat demand (case **b**, kv-only, from `WarmtevraagLeuven_v1.ipynb`), to get a rough sense of the area such a field would need at city scale — before any detailed thermal-interference study.

**Fixed assumptions** (back-of-envelope, not a detailed design):
- Borehole depth fixed at 100 m, spacing fixed at 6 m x 6 m, rectangular field.
- Simulation period: 40 years.
- Ground properties: derived from real DOV borehole/stratigraphy data for Leuven (via `pydov`), combined with literature thermal-conductivity values per formation — this is the weakest link in the chain (a literature lookup, not a direct measurement) and is flagged explicitly below.
- Borehole thermal resistance (Rb) and fluid temperature limits: typical literature defaults, not case-specific.
- **GSHP efficiency**: not all building heat comes from the ground — a heat pump also draws electricity. We convert building demand to ground load using GHEtool's own default COP=5 (`ground_extraction = heating * (1 - 1/COP)`), not the raw building demand directly.
- Heating only (case b has no cooling load).
- The field size itself is an **extrapolated** estimate (see the sizing section below for why, and how far the extrapolation reaches) — not an exact GHEtool solve.

Depends on `load_profile_case_b.parquet`, produced by `WarmtevraagLeuven_v1.ipynb`. See `BorefieldSizingLeuven_v1_case_a.ipynb` for the worst-worst-case (kv + industry) result for comparison.


In [1]:
import pathlib

import geopandas as gpd
import GHEtool as ghe
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from owslib.fes2 import And, PropertyIsGreaterThanOrEqualTo, PropertyIsEqualTo
from pydov.search.boring import BoringSearch
from pydov.search.interpretaties import FormeleStratigrafieSearch

DATA_DIR = pathlib.Path(r"C:\Workdir\Develop\ghetool\UrbanLab")

DEPTH = 100.0  # m, fixed borehole depth
SPACING = 6.0  # m, fixed spacing between boreholes
SIMULATION_PERIOD = 40  # years


## Ground properties from DOV

DOV doesn't expose a simple point-query API for thermal conductivity. Instead, we pull real formal-stratigraphy interpretations of deep boreholes in Leuven via `pydov`, clip each borehole's layers to the top 100 m, and combine them with literature thermal-conductivity values per geological formation (Flanders shallow-geothermal conductivity study). This gives a defensible, data-grounded average - not a measured value.


In [2]:
# Deep boreholes (>= 90 m) in the municipality of Leuven, so their logs cover most of our 0-100 m window.
boring_query = And([PropertyIsEqualTo('gemeente', 'Leuven'), PropertyIsGreaterThanOrEqualTo('diepte_boring_tot', '90')])
boringen = BoringSearch().search(query=boring_query, return_fields=['pkey_boring', 'x', 'y', 'diepte_boring_tot'])
print(f"{len(boringen)} deep boreholes found in Leuven")

strat_query = And([PropertyIsEqualTo('gemeente', 'Leuven'), PropertyIsGreaterThanOrEqualTo('diepte_tot_m', '90')])
strat_layers = FormeleStratigrafieSearch().search(
    query=strat_query,
    return_fields=['pkey_interpretatie', 'diepte_laag_van', 'diepte_laag_tot', 'lid1'],
)
print(f"{strat_layers['pkey_interpretatie'].nunique()} borehole stratigraphy logs, {len(strat_layers)} layers")
strat_layers.head()


[000/001] 

.

98 deep boreholes found in Leuven


[000/001] 

.

...

c

c

c

30 borehole stratigraphy logs, 137 layers


,pkey_interpretatie,diepte_laag_van,diepte_laag_tot,lid1
0,https://www.dov.vlaanderen.be/data/interpretat...,0.0,10.0,Q
1,https://www.dov.vlaanderen.be/data/interpretat...,10.0,60.0,Ld
2,https://www.dov.vlaanderen.be/data/interpretat...,60.0,89.0,Tt
3,https://www.dov.vlaanderen.be/data/interpretat...,89.0,118.0,LA
4,https://www.dov.vlaanderen.be/data/interpretat...,0.0,7.6,Q


In [3]:
# Literature thermal conductivity [W/(m.K)] per DOV lithostratigraphic member code, for the formations
# expected under Leuven: Formatie van Brussel (sand), Ieper Groep incl. Kortrijk member (clay),
# Formatie van Hannut and its members (sandy/clayey), Quartair (mixed, unconsolidated).
# Source: "Verantwoorde uitbouw van ondiepe geothermie in Vlaanderen" thermal conductivity study (indicative ranges).
K_LOOKUP = {
    'Q': 1.6,                                          # Quartair - mixed unconsolidated sediments
    'Br': 2.1,                                          # Formatie van Brussel - saturated sand
    'IE': 1.4,                                          # Ieper Groep - marine clay
    'KoOr': 1.4, 'KoMh': 1.4,                           # Kortrijk member (Ieper Groep) - clay
    'Hn': 1.8, 'HnGr': 1.8, 'HnLi': 1.8, 'HnWa': 1.8, 'HnHa': 1.8,  # Formatie van Hannut and members
    'Hs': 1.7, 'HsGe': 1.7, 'HsOr': 1.7,                # Formatie van Halen (minor unit near 100 m)
}
K_DEFAULT = 1.8  # generic Flanders subsurface average, used as fallback for unmapped/deep codes

strat_layers = strat_layers.copy()
strat_layers['top'] = strat_layers['diepte_laag_van'].clip(lower=0, upper=DEPTH)
strat_layers['bottom'] = strat_layers['diepte_laag_tot'].clip(lower=0, upper=DEPTH)
strat_layers['thickness'] = (strat_layers['bottom'] - strat_layers['top']).clip(lower=0)
strat_layers = strat_layers[strat_layers['thickness'] > 0]

unmapped_codes = sorted(set(strat_layers['lid1']) - set(K_LOOKUP))
if unmapped_codes:
    print(f"Codes without a literature k value, using default {K_DEFAULT} W/(m.K): {unmapped_codes}")
strat_layers['k'] = strat_layers['lid1'].map(K_LOOKUP).fillna(K_DEFAULT)

per_borehole = strat_layers.groupby('pkey_interpretatie').apply(
    lambda g: pd.Series({
        'covered_thickness': g['thickness'].sum(),
        'k_weighted_avg': np.average(g['k'], weights=g['thickness']),
    }),
    include_groups=False,
)
# Keep only boreholes whose log covers most of the 0-100 m window, so a single shallow log doesn't skew the average.
per_borehole = per_borehole[per_borehole['covered_thickness'] >= 0.9 * DEPTH]
print(f"{len(per_borehole)} boreholes cover >= 90% of the 0-{DEPTH:.0f} m window")

k_s = per_borehole['k_weighted_avg'].mean()
print(f"Average thermal conductivity over top {DEPTH:.0f} m: {k_s:.2f} W/(m.K) "
      f"(range {per_borehole['k_weighted_avg'].min():.2f}-{per_borehole['k_weighted_avg'].max():.2f} across boreholes)")


Codes without a literature k value, using default 1.8 W/(m.K): ['LA', 'Ld', 'MKR', 'S', 'Tt', 'ZE']


22 boreholes cover >= 90% of the 0-100 m window
Average thermal conductivity over top 100 m: 1.71 W/(m.K) (range 1.42-1.83 across boreholes)


In [4]:
# Ground temperature: standard Belgian assumption (not derived from DOV) - ~10.5 degC surface reference, +3 degC/100m gradient.
ground_data = ghe.GroundTemperatureGradient(k_s=k_s, T_g=10.5, gradient=3.0)
ground_data


{'type': 'Ground gradient temperature', 'Ground surface temperature [°C]': 10.5, 'Gradient [K/100m]': 3.0, 'Conductivity [W/(m·K)]': np.float64(1.7085222242017033), 'Volumetric heat capacity [MJ/(m³·K)]': 2.4}

## Load

Case b: kv-only hourly heat demand (heating only), from `WarmtevraagLeuven_v1.ipynb`. We build a monthly load (fast, used for the sizing search loop) and keep the full hourly series for a final accuracy check.


In [5]:
load_profile = pd.read_parquet(DATA_DIR / "load_profile_case_b.parquet")
load_profile.index = pd.to_datetime(load_profile.index)


def to_8760_hours(series: pd.Series) -> np.ndarray:
    """GHEtool's hourly load needs exactly 8760 values. The source is 2024 (a leap year, 8784 hours) and is
    also missing one hour to a DST spring-forward gap (8783 rows total). Drop the leap day (Feb 29, 24
    hours) and interpolate the single missing DST hour to get a clean 8760-hour year."""
    s = series[~((series.index.month == 2) & (series.index.day == 29))].sort_index()
    values = s.to_numpy()
    if len(values) == 8759:
        gap_pos = s.index.get_indexer([pd.Timestamp("2024-03-31 01:00", tz=s.index.tz)])[0] + 1
        interpolated = (values[gap_pos - 1] + values[gap_pos]) / 2
        values = np.insert(values, gap_pos, interpolated)
    assert len(values) == 8760, f"expected 8760 hourly values, got {len(values)}"
    return values


hourly_heating_building = to_8760_hours(load_profile["heat_demand_kWh"])

# Not all of the building's heat comes from the ground - a heat pump also uses electricity to drive the
# compressor. GHEtool's own default heat pump efficiency (SCOP=5 - see HourlyBuildingLoad) converts building
# demand to ground load via COP = Qh/W: ground_extraction = heating * (1 - 1/COP). We apply this conversion
# directly rather than using GHEtool's HourlyBuildingLoad class (which adds monthly temperature-dependent
# COP behaviour we don't need for a back-of-envelope estimate).
COP_HEATING = 5.0  # GHEtool's own default (SCOP), typical for a modern GSHP
hourly_extraction = hourly_heating_building * (1 - 1 / COP_HEATING)

print(f"Building heating demand: {hourly_heating_building.sum() / 1e6:.2f} GWh -> ground extraction: {hourly_extraction.sum() / 1e6:.2f} GWh (COP={COP_HEATING:.0f})")
print(f"Peak ground extraction: {hourly_extraction.max():.0f} kW")

hourly_load = ghe.HourlyGeothermalLoad(extraction_load=hourly_extraction, simulation_period=SIMULATION_PERIOD)


Building heating demand: 219.48 GWh -> ground extraction: 175.59 GWh (COP=5)


Peak ground extraction: 117159 kW


In [6]:
borefield = ghe.Borefield(ground_data=ground_data, load=hourly_load)
borefield.Rb = 0.12  # K/W, typical single U-tube - not case-specific
borefield.set_max_fluid_temperature(16)  # degC - typical default
borefield.set_min_fluid_temperature(0)  # degC - typical default


## Fixed-depth, fixed-spacing sizing loop

GHEtool has no built-in "solve for area at a fixed depth" method - `size()`/`size_L3()`/`size_L4()` all take a fixed field geometry and return the *required* depth, not the other way round.

**Why we extrapolate instead of computing the real answer directly, and what that costs us in confidence** (investigated directly in the GHEtool/pygfunction source in this repo):
- Every sample below is a genuine, from-scratch g-function calculation - `use_precalculated_dataset=True` (the default) only matters if `create_custom_dataset()` was called first, which it wasn't, so there is no lookup table involved and nothing is interpolated across field size.
- GHEtool already uses pygfunction's `method='equivalent'` solver - the fastest general-purpose method pygfunction offers, and both tools' own default. We are not missing a faster setting.
- The bottleneck is structural: that solver unconditionally builds the full N x N pairwise thermal-interaction matrix between every borehole pair (to decide which boreholes can be treated as thermally "equivalent") before any size reduction happens - O(N^2) in both time and memory, hardcoded inside pygfunction. This matches what we saw empirically (0.5s at 1,024 boreholes -> 73s at 4,096 - worse than O(N^2), since the subsequent linear solve also grows with N).
- At the scale of our actual answer (order 10^5-10^6 boreholes), that pairwise matrix alone would need on the order of 10^11-10^12 entries - hundreds of GB to TB of RAM. Direct computation at this scale isn't just slow, it's **infeasible with the current solver**, independent of any setting we could change. A real fix would mean writing new code against pygfunction's internals to exploit our field's perfect rectangular symmetry (it doesn't need generic O(N^2) clustering) - a genuine research project, out of scope here.
- A second, separate source of uncertainty: pygfunction's "equivalent" solver is *itself* an approximation (not the exact thermal solution), and its approximation error at the field sizes we sample from (and extrapolate towards) is not quantified anywhere in the codebase or its cited literature (validated up to a few thousand boreholes). So there are two compounding, unbounded uncertainties: (1) the statistical extrapolation itself, and (2) the solver's own unquantified approximation error - on top of which we then extrapolate 46-250x beyond our largest directly-computed sample. The fit is excellent *within* the sampled range (residuals < 0.03 in log space, printed below), but that says nothing about accuracy far beyond it. Treat every final borehole count in this notebook as an order-of-magnitude estimate, not a precise number.


In [7]:
# We tried the built-in optimise_borefield_configuration() optimizer too - its neural-network g-function
# acceleration doesn't extrapolate to city-scale field sizes and errored out; it also minimises borehole
# *count*, which favours elongated, unrealistic-footprint shapes over the compact square field we want here.
sample_sides = [8, 16, 24, 32, 48, 64]
samples = []
for s in sample_sides:
    borefield.create_rectangular_borefield(s, s, SPACING, SPACING, DEPTH, 1, 0.075)
    borefield.calculate_temperatures(length=DEPTH, hourly=True)
    samples.append((s * s, borefield.results.min_temperature))

sample_N = np.array([n for n, _ in samples], dtype=float)
sample_min_temp = np.array([t for _, t in samples])
T_g_ref = ground_data.calculate_Tg(DEPTH)  # undisturbed ground temp at borehole depth - the N -> infinity asymptote

x = np.log10(sample_N)
y = np.log10(T_g_ref - sample_min_temp)
slope, intercept = np.polyfit(x, y, 1)
fit_residuals = y - (intercept + slope * x)
print(f"Power-law fit quality (log10 space) - max abs residual: {np.abs(fit_residuals).max():.3f}")

number_of_boreholes_needed = 10 ** ((np.log10(T_g_ref) - intercept) / slope)
side = int(np.ceil(np.sqrt(number_of_boreholes_needed)))
number_of_boreholes = side * side
depth_needed = DEPTH  # by construction, this is where the extrapolated field reaches the Tf_min limit at 100 m
total_length = number_of_boreholes * depth_needed
footprint_area = ((side - 1) * SPACING) ** 2  # m^2, borehole-to-borehole footprint

print(f"Extrapolated minimal square field: {side} x {side} = {number_of_boreholes} boreholes")
print(f"Footprint area: {footprint_area / 1e4:.1f} ha ({footprint_area / 1e6:.2f} km^2)")
print(f"Total drilled length: {total_length / 1000:.0f} km")
print(f"Extrapolated {number_of_boreholes / sample_N.max():.0f}x beyond the largest directly-computed sample "
      f"({int(sample_N.max())} boreholes, side={sample_sides[-1]}) - treat as an order-of-magnitude estimate.")


Power-law fit quality (log10 space) - max abs residual: 0.023
Extrapolated minimal square field: 434 x 434 = 188356 boreholes
Footprint area: 675.0 ha (6.75 km^2)
Total drilled length: 18836 km
Extrapolated 46x beyond the largest directly-computed sample (4096 boreholes, side=64) - treat as an order-of-magnitude estimate.


### Sensitivity: surface temperature 12.5 degC instead of 10.5 degC

10.5 degC (standard Belgian assumption) was used above. As a sensitivity check, this reruns the same sampling + extrapolation with a 12.5 degC surface reference (gradient and k_s unchanged) to see how much that shifts the result.


In [8]:
ground_data_alt = ghe.GroundTemperatureGradient(k_s=k_s, T_g=12.5, gradient=3.0)
borefield_alt = ghe.Borefield(ground_data=ground_data_alt, load=hourly_load)
borefield_alt.Rb = 0.12
borefield_alt.set_max_fluid_temperature(16)
borefield_alt.set_min_fluid_temperature(0)

samples_alt = []
for s in sample_sides:
    borefield_alt.create_rectangular_borefield(s, s, SPACING, SPACING, DEPTH, 1, 0.075)
    borefield_alt.calculate_temperatures(length=DEPTH, hourly=True)
    samples_alt.append((s * s, borefield_alt.results.min_temperature))

sample_N_alt = np.array([n for n, _ in samples_alt], dtype=float)
sample_min_temp_alt = np.array([t for _, t in samples_alt])
T_g_ref_alt = ground_data_alt.calculate_Tg(DEPTH)

x_alt = np.log10(sample_N_alt)
y_alt = np.log10(T_g_ref_alt - sample_min_temp_alt)
slope_alt, intercept_alt = np.polyfit(x_alt, y_alt, 1)

number_of_boreholes_needed_alt = 10 ** ((np.log10(T_g_ref_alt) - intercept_alt) / slope_alt)
side_alt = int(np.ceil(np.sqrt(number_of_boreholes_needed_alt)))
number_of_boreholes_alt = side_alt * side_alt
footprint_area_alt = ((side_alt - 1) * SPACING) ** 2

print(f"T_g=10.5 degC (baseline):    {side} x {side} = {number_of_boreholes} boreholes, {footprint_area / 1e4:.1f} ha")
print(f"T_g=12.5 degC (sensitivity): {side_alt} x {side_alt} = {number_of_boreholes_alt} boreholes, {footprint_area_alt / 1e4:.1f} ha")
print(f"Change: {(footprint_area_alt / footprint_area - 1) * 100:+.1f}% footprint area for a +2 degC warmer surface reference")


T_g=10.5 degC (baseline):    434 x 434 = 188356 boreholes, 675.0 ha
T_g=12.5 degC (sensitivity): 398 x 398 = 158404 boreholes, 567.4 ha
Change: -15.9% footprint area for a +2 degC warmer surface reference


## Comparison to Leuven centrum's area

Re-fetch the same WFS heat-demand layer and Leuven-centrum filter used in `WarmtevraagLeuven_v1.ipynb`, and compute its area directly from the polygon geometry (rather than a hard-coded figure).


In [9]:
wfs_url = (
    "https://www.mercator.vlaanderen.be/raadpleegdienstenmercatorpubliek/wfs?service=WFS&version=2.0.0"
    "&request=GetFeature&typeNames=er:er_wrmtk_wvrg_statsec_2023&outputFormat=application/json"
)
gdf = gpd.read_file(wfs_url)
verbruik_leuven = gdf[gdf.statsec_code.str.startswith("24062A")]  # Leuven centrum, same scope as the load notebook

leuven_area = verbruik_leuven.geometry.area.sum()  # m^2, native CRS (Lambert72, EPSG:31370) is already in meters
pct_of_leuven = footprint_area / leuven_area * 100

print(f"Leuven centrum area: {leuven_area / 1e4:.1f} ha ({leuven_area / 1e6:.2f} km^2)")
print(f"Borefield footprint: {footprint_area / 1e4:.2f} ha ({footprint_area / 1e6:.3f} km^2)")
print(f"Borefield footprint is {pct_of_leuven:.2f}% of Leuven centrum's area")


Leuven centrum area: 569.7 ha (5.70 km^2)
Borefield footprint: 674.96 ha (6.750 km^2)
Borefield footprint is 118.48% of Leuven centrum's area


## Map

Leuven centrum's boundary with the sized borefield's true-scale footprint overlaid at its centroid, so the size difference is visible directly. A summary annotation gives the key numbers.


In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D

COLOR_PAGE = "#f9f9f7"
COLOR_SURFACE = "#fcfcfb"
COLOR_CITY_FILL = "#b7d3f6"
COLOR_CITY_EDGE = "#5f7ea8"
COLOR_FIELD_FILL = "#d03b3b"
COLOR_FIELD_EDGE = "#7a1f1f"
COLOR_TEXT_PRIMARY = "#0b0b0b"
COLOR_TEXT_SECONDARY = "#52514e"
COLOR_MUTED = "#898781"

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Segoe UI", "Arial", "DejaVu Sans"]

city_outline = verbruik_leuven.geometry.union_all()  # dissolve sub-sectors into one clean silhouette
centroid = city_outline.centroid
width = height = (side - 1) * SPACING

fig, ax = plt.subplots(figsize=(9, 9.8), facecolor=COLOR_PAGE)
ax.set_facecolor(COLOR_SURFACE)

gpd.GeoSeries([city_outline]).plot(ax=ax, color=COLOR_CITY_FILL, edgecolor=COLOR_CITY_EDGE, linewidth=1.3, zorder=2)
rect = Rectangle((centroid.x - width / 2, centroid.y - height / 2), width, height,
                  facecolor=COLOR_FIELD_FILL, edgecolor=COLOR_FIELD_EDGE, linewidth=1.4, alpha=0.62, zorder=3)
ax.add_patch(rect)

# matplotlib does not auto-scale axes to patches, only to plotted geometries - explicitly set limits to
# whichever is larger (city or field), with a margin, so an oversized field is never silently clipped.
city_minx, city_miny, city_maxx, city_maxy = verbruik_leuven.total_bounds
field_minx, field_miny = centroid.x - width / 2, centroid.y - height / 2
field_maxx, field_maxy = centroid.x + width / 2, centroid.y + height / 2
minx, miny = min(city_minx, field_minx), min(city_miny, field_miny)
maxx, maxy = max(city_maxx, field_maxx), max(city_maxy, field_maxy)
pad = 0.10 * max(maxx - minx, maxy - miny)
ax.set_xlim(minx - pad, maxx + pad)
ax.set_ylim(miny - pad, maxy + 0.20 * (maxy - miny))
ax.set_aspect("equal")
ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

fig.suptitle("Could one equivalent collective borefield heat (cool*) Leuven centrum?", fontsize=15.5,
             fontweight="bold", color=COLOR_TEXT_PRIMARY, y=0.975, x=0.5)
fig.text(0.5, 0.94, "*cool for case c", fontsize=9.5, style="italic", color=COLOR_MUTED, ha="center")
ax.set_title("Case b - Residential only", fontsize=15, color=COLOR_TEXT_SECONDARY, pad=10)

legend_handles = [
    Line2D([0], [0], marker='s', markersize=19, markerfacecolor=COLOR_CITY_FILL, markeredgecolor=COLOR_CITY_EDGE,
           linewidth=0, label="Leuven centrum"),
    Line2D([0], [0], marker='s', markersize=19, markerfacecolor=COLOR_FIELD_FILL, markeredgecolor=COLOR_FIELD_EDGE,
           linewidth=0, alpha=0.75, label="Collective borefield"),
]
ax.legend(handles=legend_handles, loc="upper left", frameon=False, fontsize=15, labelcolor=COLOR_TEXT_SECONDARY,
          handletextpad=0.6, borderaxespad=0.5)

# scale bar
bar_len = 1000  # m
x0 = minx - pad + 0.06 * (maxx - minx + 2 * pad)
y0 = miny - pad + 0.05 * (maxy - miny + 2 * pad)
ax.plot([x0, x0 + bar_len], [y0, y0], color=COLOR_TEXT_SECONDARY, linewidth=2, solid_capstyle="butt", zorder=5)
ax.plot([x0, x0], [y0 - 40, y0 + 40], color=COLOR_TEXT_SECONDARY, linewidth=2, zorder=5)
ax.plot([x0 + bar_len, x0 + bar_len], [y0 - 40, y0 + 40], color=COLOR_TEXT_SECONDARY, linewidth=2, zorder=5)
ax.text(x0 + bar_len / 2, y0 + 90, "1 km", ha="center", va="bottom", fontsize=9, color=COLOR_TEXT_SECONDARY)

# hero stat callout
ax.text(0.985, 0.155, f"{pct_of_leuven:.0f}%", transform=ax.transAxes, ha="right", va="bottom",
        fontsize=40, fontweight="bold", color=COLOR_FIELD_EDGE, zorder=6)
ax.text(0.985, 0.135, "of Leuven centrum's area", transform=ax.transAxes, ha="right", va="top",
        fontsize=13, fontweight="bold", color=COLOR_FIELD_EDGE, zorder=6)
ax.text(0.985, 0.02, f"{number_of_boreholes:,} boreholes  ·  {footprint_area / 1e6:.1f} km² footprint  ·  100 m deep  ·  6 m spacing",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=11.5, color=COLOR_TEXT_PRIMARY,
        fontweight="bold", zorder=6)

fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(DATA_DIR / "map_case_b.png", dpi=150, facecolor=COLOR_PAGE)
plt.show()
